#### Visão Geral
##### Schema : silver
##### Table : case_erp_pedidos_itens

| Detalhe | Informação |
|---------|------------|
| Criado Originalmente Por | Wellikiandre Bosich |
| Tabela de Dados de Saída | `{environment}.silver.case_erp_pedidos_itens` |
| Origem Fonte de Dados de Entrada | Camada bronze |
| Destino Fonte de Dados de Saída | Camada silver |

#### Histórico

| Data       | Desenvolvido Por         | Motivo                                         |
|:----------:|--------------------------|-----------------------------------------------|
| 04/06/2026 | Wellikiandre Bosich    | Criação do notebook e unificação das métricas de itens de pedidos na Silver. |

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
sistema = 'case'
table_name = 'erp_pedidos_itens'
input_path = f"{var_bronze}/{sistema}/{table_name}/data"
output_path_data = f"{var_silver}/{sistema}/{table_name}/data"
table_name_schema = f'{var_environment}.{var_silver_schema}.{sistema}_{table_name}'

In [ ]:
from pyspark.sql.functions import col, md5, concat_ws
from pyspark.sql.types import DecimalType

df_bronze = spark.read.format("delta").load(input_path)

df_clean = (
    df_bronze
    .withColumn("valor_unitario", col("valor_unitario").cast(DecimalType(10, 2)))
    .select(
        col("id_pedido").cast("integer").alias("id_pedido"),
        col("id_produto").cast("string").alias("id_produto"),
        col("quantidade").cast("integer").alias("quantidade"),
        col("valor_unitario").alias("preco_unitario")
    )
    .filter(col("id_pedido").isNotNull() & col("id_produto").isNotNull())
    .withColumn("id_item_pedido", md5(concat_ws("-", col("id_pedido"), col("id_produto"))))
)

In [ ]:
process_data(
    df_write=df_clean,
    tipo_carga='delta',
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    chave_clusterby=['id_pedido'],
    chave_upsert='id_item_pedido'
)